# 04 — BRINDA inflammation correction and negative-control sensitivity

**Purpose:** strengthen the iron-deficiency analysis by correcting ferritin for inflammation (BRINDA regression-correction, CRP-only because the ENANI-2019 protocol does not include AGP), and validate the design by testing iron supplementation against an *a priori* unrelated negative-control outcome (accidental hospitalisation in the preceding 12 months).

**Components produced:**
1. BRINDA-corrected ferritin (continuous) and BRINDA-corrected iron deficiency (ferritin_BRINDA < 12 ng/mL), with diagnostics on the correction.
2. Adjusted Poisson (binary) and OLS (continuous) regression on the BRINDA outcomes with HC1 robust SE; outer bootstrap B = 2,000 with MICE imputation per resample.
3. BRINDA × breastfeeding stratified analysis (exclusive BF, mixed BF, non-BF) and formal interaction test on `iron_deficiency_BRINDA`.
4. Negative-control sensitivity: modified Poisson regression of accidental hospitalisation on iron supplementation with the 26 confounders; expected null association supports the design, a positive association would flag residual confounding by healthcare-seeking behaviour.
5. Forest plots and CSV/markdown report compiled to `04_brinda_negative_control_results/`.


In [ ]:
import warnings
import time
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_PATH = Path('../Data/data_crianca_calib_anon.csv')
RNG = np.random.default_rng(42)
B = 2000
Z95 = 1.959964
print(f'Configuration: B = {B}, MICE-imputed covariates within bootstrap, HC1 analytical CIs.')


## 1. Data and variable construction

Same 26 confounders as Notebook 03, plus a three-level feeding stratification (exclusive breastfeeding, mixed breastfeeding, non-breastfed) for the BRINDA × breastfeeding analysis.


In [ ]:
raw = pd.read_csv(DATA_PATH, low_memory=False)
raw['age_months'] = raw['b05a_idade_em_meses'].astype(str).str.extract(r'(\d+)').astype(float)
df = raw[(raw['age_months'] >= 6) & (raw['age_months'] <= 24)].copy()

df['iron_any'] = (df['vd_supl1_com_ferro'] == 'Sim').astype(int)
df['male'] = (df['b02_sexo'] == 'Masculino').astype(int)
df['race_preta'] = (df['d01_cor'] == 'Preta').astype(int)
df['race_parda'] = df['d01_cor'].isin(['Parda (mulata, cabocla, cafuza, mameluca ou mestiça)', 'Parda']).astype(int)
df['reg_norte'] = (df['a00_regiao'] == 'Norte').astype(int)
df['reg_nordeste'] = (df['a00_regiao'] == 'Nordeste').astype(int)
df['reg_sul'] = (df['a00_regiao'] == 'Sul').astype(int)
df['reg_co'] = (df['a00_regiao'] == 'Centro-Oeste').astype(int)
df['urban'] = (df['a11_situacao'] == 'Urbano').astype(int)
df['ien'] = pd.to_numeric(df['vd_ien_quintos'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
df['ebia_leve'] = (df['vd_ebia_categ'] == 'Insegurança leve').astype(int)
df['ebia_mod'] = (df['vd_ebia_categ'] == 'Insegurança moderada').astype(int)
df['ebia_grave'] = (df['vd_ebia_categ'] == 'Insegurança grave').astype(int)
df['esgoto_adeq'] = df['p10_esgoto'].isin([
    'Rede geral de esgoto ou pluvial', 'Fossa séptica ligada a rede', 'Fossa séptica não ligada a rede',
]).astype(int)
df['agua_rede'] = (df['p11_agua'] == 'Rede geral de distribuição').astype(int)
df['cesarean'] = df['h04_parto'].isin(['Cesariana agendada (eletiva)', 'Cesariana de urgência (não agendada)']).astype(int)
df['gest_weeks'] = pd.to_numeric(df['h01_semanas_gravidez'], errors='coerce')
df['birth_weight'] = pd.to_numeric(df['h02_peso'], errors='coerce')
df['prenatal_visits'] = pd.to_numeric(df['k05_prenatal_consultas'], errors='coerce')
df['maternal_age'] = pd.to_numeric(df['bb04_idade_da_mae'], errors='coerce')
df['breastfed'] = (df['e01_leite_peito'] == 'Sim').astype(int)
df['formula'] = (df['e10_formula_infantil'] == 'Sim').astype(int)
df['cow_milk'] = ((df['e06_leite_vaca_po'] == 'Sim') | (df['e07_leite_vaca_liquido'] == 'Sim')).astype(int)

import re
def _educ_band(text):
    if pd.isna(text):
        return 'Unknown'
    t = str(text)
    if 'Sem estudo' in t:
        return 'Less than primary'
    if 'fundamental' in t.lower():
        m = re.search(r'(\d+)', t)
        if m and int(m.group(1)) >= 8:
            return 'Primary completed'
        return 'Less than primary'
    if 'ensino médio' in t.lower():
        if '3°ano' in t or '3° ano' in t:
            return 'Secondary completed'
        return 'Primary completed'
    if 'superior' in t.lower():
        return 'Tertiary or above'
    return 'Unknown'
df['_education_band'] = df['j10_serie'].apply(_educ_band)
df['educ_primary'] = (df['_education_band'] == 'Primary completed').astype(int)
df['educ_secondary'] = (df['_education_band'] == 'Secondary completed').astype(int)
df['educ_tertiary'] = (df['_education_band'] == 'Tertiary or above').astype(int)

df['ferritin'] = pd.to_numeric(df['vd_ferri_final'], errors='coerce')
df['crp'] = pd.to_numeric(df['vd_pcr_final'], errors='coerce')

# Three-level feeding stratification
df['exclusive_bf'] = ((df['breastfed'] == 1) & (df['formula'] == 0) & (df['cow_milk'] == 0)).astype(int)
df['mixed_bf'] = ((df['breastfed'] == 1) & ((df['formula'] == 1) | (df['cow_milk'] == 1))).astype(int)
df['non_bf'] = (df['breastfed'] == 0).astype(int)

CONFOUNDERS = [
    'age_months', 'male', 'race_preta', 'race_parda',
    'reg_norte', 'reg_nordeste', 'reg_sul', 'reg_co', 'urban',
    'ien', 'ebia_leve', 'ebia_mod', 'ebia_grave',
    'esgoto_adeq', 'agua_rede', 'cesarean',
    'gest_weeks', 'birth_weight', 'prenatal_visits', 'maternal_age',
    'breastfed', 'formula', 'cow_milk',
    'educ_primary', 'educ_secondary', 'educ_tertiary',
]
MISSING_PRONE = ['gest_weeks', 'birth_weight', 'prenatal_visits', 'maternal_age', 'ien']

print(f'Analytic sample: {len(df):,}')
print(f'Iron-exposed: {int(df.iron_any.sum()):,} ({df.iron_any.mean()*100:.1f}%)')
print(f'Confounders: {len(CONFOUNDERS)}')
print(f'Feeding strata: exclusive_bf={int(df.exclusive_bf.sum())}, mixed_bf={int(df.mixed_bf.sum())}, non_bf={int(df.non_bf.sum())}')
print(f'CRP available: {int(df.crp.notna().sum())}; ferritin available: {int(df.ferritin.notna().sum())}; both: {int((df.crp.notna() & df.ferritin.notna()).sum())}')


## 2. BRINDA inflammation correction (CRP-only, regression method)

The BRINDA regression-correction adjusts ferritin downward in individuals with elevated inflammatory markers (Namaste et al. 2017). With AGP unavailable in ENANI-2019, the simplified CRP-only correction is used:

ln(ferritin)_corrected_i = ln(ferritin_i) − β · max(0, ln(CRP_i) − ln(CRP_ref))

where β is the slope from the population-level regression ln(ferritin) ~ ln(CRP) and CRP_ref is the 10th percentile of CRP (the low-inflammation reference). Only individuals with elevated CRP (above the reference) receive a downward correction. Ferritin missingness or CRP missingness propagates to BRINDA-corrected ferritin and to the BRINDA-corrected iron-deficiency indicator.


In [ ]:
# Fit population regression of log(ferritin) on log(CRP) using complete cases
both_avail = df[['ferritin', 'crp']].dropna().copy()
both_avail['ln_ferritin'] = np.log(both_avail['ferritin'].clip(lower=0.1))
both_avail['ln_crp'] = np.log(both_avail['crp'].clip(lower=0.01))

brinda_model = smf.ols('ln_ferritin ~ ln_crp', data=both_avail).fit()
beta_crp = float(brinda_model.params['ln_crp'])

crp_p10 = float(both_avail['crp'].quantile(0.10))
ln_crp_ref = np.log(max(0.01, crp_p10))

print(f'BRINDA regression (n = {len(both_avail)}):')
print(f'  ln(ferritin) ~ ln(CRP)')
print(f'  beta_CRP = {beta_crp:+.4f}')
print(f'  CRP 10th percentile (reference) = {crp_p10:.3f} mg/L')
print(f'  R-squared = {brinda_model.rsquared:.4f}')

# Apply correction
df['_ln_ferritin'] = np.log(df['ferritin'].clip(lower=0.1))
df['_ln_crp'] = np.log(df['crp'].clip(lower=0.01))
correction = beta_crp * np.maximum(0, df['_ln_crp'] - ln_crp_ref)
df['ferritin_brinda'] = np.exp(df['_ln_ferritin'] - correction)
df.loc[df['ferritin'].isna() | df['crp'].isna(), 'ferritin_brinda'] = np.nan

df['iron_deficiency_brinda'] = (df['ferritin_brinda'] < 12).where(df['ferritin_brinda'].notna(), other=pd.NA).astype('Int64')
df['iron_deficiency_uncorrected'] = (df['ferritin'] < 12).where(df['ferritin'].notna(), other=pd.NA).astype('Int64')

# Diagnostics
n_brinda = int(df['ferritin_brinda'].notna().sum())
id_brinda = int(df.loc[df['iron_deficiency_brinda'].notna(), 'iron_deficiency_brinda'].sum())
id_uncorr = int(df.loc[df['iron_deficiency_uncorrected'].notna(), 'iron_deficiency_uncorrected'].sum())

# Reclassification table (in the n_brinda sample with both ferritin and CRP)
sub = df.dropna(subset=['ferritin_brinda', 'iron_deficiency_uncorrected']).copy()
reclass = pd.crosstab(sub['iron_deficiency_uncorrected'].astype(int), sub['iron_deficiency_brinda'].astype(int),
                      rownames=['Uncorrected'], colnames=['BRINDA-corrected'])

print()
print(f'Ferritin: median uncorrected = {df.ferritin.median():.2f}, median BRINDA = {df.ferritin_brinda.median():.2f}')
print(f'Iron deficiency events (uncorrected, ferritin<12): {id_uncorr} / {int(df.iron_deficiency_uncorrected.notna().sum())}')
print(f'Iron deficiency events (BRINDA, ferritin_brinda<12): {id_brinda} / {n_brinda}')
print()
print('Reclassification (BRINDA upgrades cases that uncorrected ferritin missed because of inflammation-elevated values):')
print(reclass)


## 3. Helper functions and adjusted analysis

Same MICE / bootstrap / fit machinery as Notebook 03, applied to two outcomes — `ferritin_brinda` (continuous) and `iron_deficiency_brinda` (binary).


In [ ]:
def mice_impute_covariates(df_in, vars_to_impute=MISSING_PRONE, max_iter=10, random_state=None):
    df_out = df_in.copy()
    needed = [v for v in vars_to_impute if df_out[v].isna().any()]
    if not needed:
        return df_out
    aux = [v for v in CONFOUNDERS if v not in vars_to_impute and df_out[v].isna().sum() == 0]
    matrix = df_out[needed + aux].astype(float)
    imp = IterativeImputer(estimator=BayesianRidge(), max_iter=max_iter, random_state=random_state, sample_posterior=False)
    imputed = imp.fit_transform(matrix)
    df_out[needed] = imputed[:, : len(needed)]
    return df_out

def fit_continuous(df_in, outcome, exposure='iron_any'):
    sub = df_in[[outcome, exposure] + CONFOUNDERS].dropna(subset=[outcome, exposure])
    if len(sub) < 100:
        return None
    rhs = exposure + ' + ' + ' + '.join(CONFOUNDERS)
    return smf.ols(f'{outcome} ~ {rhs}', data=sub).fit(cov_type='HC1')

def fit_binary(df_in, outcome, exposure='iron_any'):
    sub = df_in[[outcome, exposure] + CONFOUNDERS].dropna(subset=[outcome, exposure]).copy()
    sub[outcome] = sub[outcome].astype(int)
    if sub[outcome].sum() < 5 or len(sub) < 100:
        return None
    rhs = exposure + ' + ' + ' + '.join(CONFOUNDERS)
    try:
        return smf.glm(f'{outcome} ~ {rhs}', data=sub, family=sm.families.Poisson()).fit(cov_type='HC1', disp=False)
    except Exception:
        return None

def boot_pvalue(coefs, null=0.0):
    arr = np.asarray(coefs)
    p_lo = float(np.mean(arr <= null))
    p_hi = float(np.mean(arr >= null))
    return float(min(1.0, 2 * min(p_lo, p_hi)))

def outer_bootstrap(df_in, outcome, kind, B=B, exposure='iron_any', seed=42):
    rng = np.random.default_rng(seed)
    base = df_in[[outcome, exposure] + CONFOUNDERS].dropna(subset=[outcome, exposure]).copy()
    n = len(base)
    coefs = []
    fitter = fit_continuous if kind == 'continuous' else fit_binary
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        sample = base.iloc[idx].reset_index(drop=True)
        sample = mice_impute_covariates(sample, random_state=int(b))
        m = fitter(sample, outcome, exposure=exposure)
        if m is not None and exposure in m.params.index:
            coefs.append(float(m.params[exposure]))
    return np.asarray(coefs)

# Single MICE pass for analytical estimation
df_imputed = mice_impute_covariates(df, random_state=42)

# Analytical primary
brinda_rows = []
for outcome, kind, label in [
    ('ferritin_brinda', 'continuous', 'Ferritin (BRINDA-corrected, ng/mL)'),
    ('iron_deficiency_brinda', 'binary', 'Iron deficiency (BRINDA-corrected, ferritin<12)'),
]:
    fitter = fit_continuous if kind == 'continuous' else fit_binary
    m = fitter(df_imputed, outcome)
    if m is None:
        continue
    coef = float(m.params['iron_any']); se = float(m.bse['iron_any']); p = float(m.pvalues['iron_any'])
    if kind == 'continuous':
        est = coef
        ci_lo, ci_hi = coef - Z95 * se, coef + Z95 * se
        measure = 'beta'
    else:
        est = float(np.exp(coef))
        ci_lo, ci_hi = float(np.exp(coef - Z95 * se)), float(np.exp(coef + Z95 * se))
        measure = 'PR'
    brinda_rows.append({
        'Outcome': label, 'Kind': kind, 'Measure': measure,
        'Estimate': est, 'CI_lo_anal': ci_lo, 'CI_hi_anal': ci_hi,
        'p_anal': p, 'N': int(m.nobs), '_var': outcome,
    })

# Bootstrap
print('Running bootstrap for BRINDA outcomes...')
boot_results = {}
t_start = time.time()
for outcome, kind, _ in [('ferritin_brinda', 'continuous', ''), ('iron_deficiency_brinda', 'binary', '')]:
    t0 = time.time()
    coefs = outer_bootstrap(df, outcome, kind, B=B, seed=42)
    if len(coefs) == 0:
        boot_results[outcome] = None
        continue
    if kind == 'binary':
        prs = np.exp(coefs)
        boot_results[outcome] = {
            'CI_lo_boot': float(np.percentile(prs, 2.5)),
            'CI_hi_boot': float(np.percentile(prs, 97.5)),
            'p_boot': boot_pvalue(coefs, null=0.0),
            'coefs': coefs,
        }
    else:
        boot_results[outcome] = {
            'CI_lo_boot': float(np.percentile(coefs, 2.5)),
            'CI_hi_boot': float(np.percentile(coefs, 97.5)),
            'p_boot': boot_pvalue(coefs, null=0.0),
            'coefs': coefs,
        }
    print(f'  {outcome:30s}  {len(coefs)} reps  {time.time()-t0:5.1f}s  p_boot={boot_results[outcome]["p_boot"]:.4f}')
print(f'Total bootstrap time: {(time.time()-t_start)/60:.1f} min')

# Combined table
brinda_combined = []
for r in brinda_rows:
    var = r['_var']; boot = boot_results.get(var)
    brinda_combined.append({
        'Outcome': r['Outcome'], 'Measure': r['Measure'], 'Estimate': r['Estimate'],
        'CI 95% (analytical)': f"[{r['CI_lo_anal']:.4f}, {r['CI_hi_anal']:.4f}]",
        'p (analytical)': r['p_anal'],
        'CI 95% (bootstrap)': f"[{boot['CI_lo_boot']:.4f}, {boot['CI_hi_boot']:.4f}]" if boot else '—',
        'p (bootstrap)': boot['p_boot'] if boot else np.nan,
        'N': r['N'],
    })
brinda_table = pd.DataFrame(brinda_combined)
brinda_table


## 4. BRINDA × breastfeeding — stratified analysis and formal interaction

Two complementary tests: (i) prevalence ratios for `iron_deficiency_brinda` per feeding stratum (exclusive BF, mixed BF, non-BF) to surface where the iron association is concentrated; (ii) a formal interaction term (iron × breastfed) on the same outcome with the 25 confounders that exclude breastfed.


In [ ]:
# (i) Stratified PRs (with HC1, no bootstrap to keep runtime modest; descriptive)
strat_rows = []
for stratum, mask_var in [('Exclusive BF', 'exclusive_bf'), ('Mixed BF', 'mixed_bf'), ('Non-BF', 'non_bf')]:
    sub = df_imputed[df_imputed[mask_var] == 1].copy()
    sub_data = sub[['iron_deficiency_brinda', 'iron_any'] + CONFOUNDERS].dropna(subset=['iron_deficiency_brinda', 'iron_any'])
    sub_data['iron_deficiency_brinda'] = sub_data['iron_deficiency_brinda'].astype(int)
    if len(sub_data) < 100 or sub_data['iron_deficiency_brinda'].sum() < 10:
        strat_rows.append({'Stratum': stratum, 'N': len(sub_data), 'Events': int(sub_data['iron_deficiency_brinda'].sum()),
                           'PR': np.nan, 'CI 95%': '—', 'p': np.nan})
        continue
    # Drop perfectly collinear stratum-defining feeding variables to avoid singular design
    drop_vars = {'exclusive_bf': ['breastfed', 'formula', 'cow_milk'],
                 'mixed_bf': ['breastfed'],
                 'non_bf': ['breastfed']}.get(mask_var, [])
    use_conf = [c for c in CONFOUNDERS if c not in drop_vars]
    rhs = 'iron_any + ' + ' + '.join(use_conf)
    try:
        m = smf.glm(f'iron_deficiency_brinda ~ {rhs}', data=sub_data, family=sm.families.Poisson()).fit(cov_type='HC1', disp=False)
        coef = float(m.params['iron_any']); se = float(m.bse['iron_any']); p = float(m.pvalues['iron_any'])
        pr = float(np.exp(coef))
        ci_lo, ci_hi = float(np.exp(coef - Z95*se)), float(np.exp(coef + Z95*se))
        strat_rows.append({'Stratum': stratum, 'N': len(sub_data), 'Events': int(sub_data['iron_deficiency_brinda'].sum()),
                           'PR': pr, 'CI 95%': f'[{ci_lo:.3f}, {ci_hi:.3f}]', 'p': p})
    except Exception as e:
        strat_rows.append({'Stratum': stratum, 'N': len(sub_data), 'Events': int(sub_data['iron_deficiency_brinda'].sum()),
                           'PR': np.nan, 'CI 95%': f'fit failed: {e}', 'p': np.nan})
strat_table = pd.DataFrame(strat_rows)
print('Stratified BRINDA-corrected iron deficiency by feeding stratum:')
print(strat_table.to_string(index=False))

# (ii) Formal interaction on iron_deficiency_brinda — iron × breastfed
INT_CONF = [c for c in CONFOUNDERS if c != 'breastfed']
sub = df_imputed[['iron_deficiency_brinda', 'iron_any', 'breastfed'] + INT_CONF].dropna(subset=['iron_deficiency_brinda', 'iron_any', 'breastfed']).copy()
sub['iron_deficiency_brinda'] = sub['iron_deficiency_brinda'].astype(int)
sub['iron_x_bf'] = sub['iron_any'] * sub['breastfed']
rhs_int = 'iron_any + breastfed + iron_x_bf + ' + ' + '.join(INT_CONF)
m_int = smf.glm(f'iron_deficiency_brinda ~ {rhs_int}', data=sub, family=sm.families.Poisson()).fit(cov_type='HC1', disp=False)
int_coef = float(m_int.params['iron_x_bf']); int_se = float(m_int.bse['iron_x_bf']); int_p = float(m_int.pvalues['iron_x_bf'])

# Bootstrap interaction with B=2000, MICE per resample
print('Bootstrapping iron × BF interaction on BRINDA-corrected iron deficiency...')
def boot_int_brinda(df_in, B=B, seed=44):
    rng = np.random.default_rng(seed)
    base = df_in[['iron_deficiency_brinda', 'iron_any', 'breastfed'] + INT_CONF].dropna(subset=['iron_deficiency_brinda', 'iron_any', 'breastfed']).copy()
    base['iron_deficiency_brinda'] = base['iron_deficiency_brinda'].astype(int)
    n = len(base); coefs = []
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        s = base.iloc[idx].reset_index(drop=True)
        s = mice_impute_covariates(s, random_state=int(b))
        s['iron_x_bf'] = s['iron_any'] * s['breastfed']
        try:
            mm = smf.glm(f'iron_deficiency_brinda ~ {rhs_int}', data=s, family=sm.families.Poisson()).fit(disp=False)
            coefs.append(float(mm.params['iron_x_bf']))
        except Exception:
            continue
    return np.asarray(coefs)

t0 = time.time()
int_coefs = boot_int_brinda(df, B=B, seed=44)
boot_int_lo = float(np.percentile(int_coefs, 2.5))
boot_int_hi = float(np.percentile(int_coefs, 97.5))
boot_int_p = boot_pvalue(int_coefs, null=0.0)
print(f'Iron × BF on iron_deficiency_brinda: coef={int_coef:+.4f} (SE {int_se:.4f}), '
      f'analytical p={int_p:.4f}, bootstrap CI=[{boot_int_lo:+.4f}, {boot_int_hi:+.4f}], p_boot={boot_int_p:.4f}, '
      f'time={time.time()-t0:.1f}s')

interaction_brinda_table = pd.DataFrame([{
    'Outcome': 'Iron deficiency (BRINDA-corrected)',
    'Interaction coef (log-PR)': int_coef,
    'CI 95% (analytical)': f'[{int_coef-Z95*int_se:+.4f}, {int_coef+Z95*int_se:+.4f}]',
    'p (analytical)': int_p,
    'CI 95% (bootstrap)': f'[{boot_int_lo:+.4f}, {boot_int_hi:+.4f}]',
    'p (bootstrap)': boot_int_p,
    'N': int(m_int.nobs),
}])
interaction_brinda_table


## 5. Negative-control sensitivity — accidental hospitalisation

A negative-control outcome (Lipsitch et al. 2010) shares the same plausible confounders as the exposure but lacks any plausible mechanistic link to it. Accidental hospitalisation in the preceding twelve months is influenced by socioeconomic and environmental factors but cannot plausibly be reduced by iron supplementation. A null association after adjustment supports the design; a non-null association would flag residual confounding (e.g., by healthcare-seeking behaviour, which simultaneously raises the probability of receiving PNSF iron and being hospitalised for a recognised cause).

Modified Poisson regression of `hosp_accident` on `iron_any` plus the 26 confounders, with HC1 robust SE and outer bootstrap (B = 2,000).


In [ ]:
# Build accident-hospitalisation outcome.
# h216_internado_nao = 'Sim' means NOT hospitalised in the past year (3,989 / 4,601).
# h213_internado_acidente = 'Sim' if accidental hospitalisation occurred (22 / 4,601).
df['hosp_any'] = (df['h216_internado_nao'] == 'Não').astype('Int64')
df.loc[df['h216_internado_nao'].isna(), 'hosp_any'] = pd.NA
df['hosp_accident'] = (df['h213_internado_acidente'] == 'Sim').astype('Int64')
df.loc[df['h216_internado_nao'].isna(), 'hosp_accident'] = pd.NA

n_avail = int(df['hosp_accident'].notna().sum())
n_events = int(df.loc[df['hosp_accident'].notna(), 'hosp_accident'].sum())
print(f'Accidental hospitalisation: events = {n_events} / {n_avail} ({100*n_events/n_avail:.2f}%)')
print(f'  Among iron-exposed: {int(((df.iron_any==1)&(df.hosp_accident==1)).sum())} / {int(((df.iron_any==1)&(df.hosp_accident.notna())).sum())}')
print(f'  Among unexposed:    {int(((df.iron_any==0)&(df.hosp_accident==1)).sum())} / {int(((df.iron_any==0)&(df.hosp_accident.notna())).sum())}')

# Need to also rebuild df_imputed to include hosp_accident (covariates already imputed in previous df_imputed)
df_imputed['hosp_accident'] = df['hosp_accident'].values

# Analytical
m_neg = fit_binary(df_imputed, 'hosp_accident')
if m_neg is not None:
    coef = float(m_neg.params['iron_any']); se = float(m_neg.bse['iron_any']); p = float(m_neg.pvalues['iron_any'])
    pr = float(np.exp(coef))
    ci_lo, ci_hi = float(np.exp(coef - Z95*se)), float(np.exp(coef + Z95*se))
    print(f'\nAnalytical: PR = {pr:.3f}  [{ci_lo:.3f}, {ci_hi:.3f}]  p = {p:.4f}')

    # Bootstrap
    print('Bootstrapping negative control...')
    t0 = time.time()
    neg_coefs = outer_bootstrap(df, 'hosp_accident', 'binary', B=B, seed=45)
    if len(neg_coefs) > 0:
        prs = np.exp(neg_coefs)
        neg_boot = {
            'CI_lo': float(np.percentile(prs, 2.5)),
            'CI_hi': float(np.percentile(prs, 97.5)),
            'p_boot': boot_pvalue(neg_coefs, null=0.0),
        }
        print(f'Bootstrap (B = {len(neg_coefs)}): PR_boot CI = [{neg_boot["CI_lo"]:.3f}, {neg_boot["CI_hi"]:.3f}]  p_boot = {neg_boot["p_boot"]:.4f}  time={time.time()-t0:.1f}s')

    negative_control_table = pd.DataFrame([{
        'Outcome': 'Accidental hospitalisation (negative control)',
        'Measure': 'PR',
        'Estimate': pr,
        'CI 95% (analytical)': f'[{ci_lo:.3f}, {ci_hi:.3f}]',
        'p (analytical)': p,
        'CI 95% (bootstrap)': f'[{neg_boot["CI_lo"]:.3f}, {neg_boot["CI_hi"]:.3f}]' if len(neg_coefs)>0 else '—',
        'p (bootstrap)': neg_boot['p_boot'] if len(neg_coefs)>0 else np.nan,
        'N': int(m_neg.nobs),
        'Events': int(df_imputed.loc[df_imputed['hosp_accident'].notna(), 'hosp_accident'].sum()),
    }])
    negative_control_table

    # Decision rule
    decision = 'NULL: design supported (no residual healthcare-seeking confounding detected via this control)' if neg_boot['p_boot'] > 0.05 else 'NON-NULL: residual confounding flag — investigate'
    print(f'\nNegative-control decision: {decision}')
else:
    negative_control_table = pd.DataFrame()
    print('Negative-control fit failed (likely too few events).')


## 6. Notes

- BRINDA-corrected ferritin is computed with CRP only (AGP not measured in ENANI-2019). Sensitivity to the choice of CRP reference percentile (5th, 10th, 25th) can be added if a reviewer requests it.
- The BRINDA × breastfeeding stratified analysis is descriptive (HC1 analytical CIs, no bootstrap) to keep runtime modest; the formal interaction test is bootstrapped (B = 2,000).
- The negative control assumes accidental hospitalisation has no plausible mechanistic link to iron supplementation. A non-null bootstrap p-value would not invalidate the primary findings on its own but would weaken the assumption that healthcare-seeking is fully captured by the 26 confounders.
- All adjusted models use the 26 confounders defined in Notebook 03; outcomes are never imputed.


## 7. Export

Tables and figures are written to `04_brinda_negative_control_results/` on each run. The auto-generated `REPORT.md` compiles all tables and inserts the figure references.


In [ ]:
OUTPUT_DIR = Path('./04_brinda_negative_control_results')
TABLES_DIR = OUTPUT_DIR / 'tables'
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# Save tables
brinda_table.to_csv(TABLES_DIR / 'brinda_adjusted.csv', index=False)
strat_table.to_csv(TABLES_DIR / 'brinda_stratified_by_feeding.csv', index=False)
interaction_brinda_table.to_csv(TABLES_DIR / 'brinda_iron_x_bf_interaction.csv', index=False)
if not negative_control_table.empty:
    negative_control_table.to_csv(TABLES_DIR / 'negative_control_accident.csv', index=False)

# BRINDA scatter plot: ferritin vs CRP with regression line
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(both_avail['crp'], both_avail['ferritin'], alpha=0.2, s=8, color='gray')
xs = np.linspace(both_avail['crp'].min(), both_avail['crp'].max(), 200)
ys = np.exp(brinda_model.params['Intercept'] + brinda_model.params['ln_crp'] * np.log(np.maximum(0.01, xs)))
ax.plot(xs, ys, color='C3', lw=2, label=f'log(ferritin) ~ log(CRP)\n  beta={beta_crp:+.3f}, R^2={brinda_model.rsquared:.3f}')
ax.axvline(crp_p10, color='black', linestyle='--', lw=0.8, label=f'CRP p10 = {crp_p10:.2f} (correction reference)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('CRP (mg/L)'); ax.set_ylabel('Ferritin (ng/mL)')
ax.set_title('BRINDA regression — ferritin vs CRP (log-log)')
ax.legend(loc='upper left', fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'brinda_regression_diagnostic.png', dpi=150, bbox_inches='tight')
plt.close(fig)

# Stratified forest plot
fig, ax = plt.subplots(figsize=(7, 3.5))
y = np.arange(len(strat_table))
prs = strat_table['PR'].astype(float).values
los = []; his = []
for ci in strat_table['CI 95%']:
    if ci == '—' or 'fail' in str(ci):
        los.append(np.nan); his.append(np.nan)
    else:
        a, b = ci.strip('[]').split(',')
        los.append(float(a.strip())); his.append(float(b.strip()))
los = np.array(los); his = np.array(his)
xerr = np.array([prs - los, his - prs])
ax.errorbar(prs, y, xerr=xerr, fmt='s', color='#c0392b', capsize=4)
ax.axvline(1, color='black', lw=0.7, linestyle='--')
ax.set_yticks(y); ax.set_yticklabels(strat_table['Stratum'])
ax.set_xlabel('Prevalence ratio (BRINDA-corrected iron deficiency)')
ax.set_title('BRINDA-corrected iron deficiency — stratified by feeding')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'brinda_stratified_by_feeding.png', dpi=150, bbox_inches='tight')
plt.close(fig)

# REPORT.md
report = []
report.append('# 04 — BRINDA inflammation correction and negative-control sensitivity\n')
report.append(f'**Sample:** Brazilian infants 6–24 mo (ENANI-2019). N = {len(df):,}; biomarker-with-CRP subsample n = {n_brinda:,}.\n')
report.append(f'**Methods:** BRINDA regression-correction (CRP-only, beta_CRP = {beta_crp:+.4f}, CRP p10 = {crp_p10:.3f} mg/L); adjusted Poisson / OLS with HC1 robust SE; outer bootstrap B = {B} with MICE imputation per resample; 26 confounders.\n')
report.append('---\n')
report.append('## BRINDA regression diagnostic\n')
report.append('![BRINDA regression](figures/brinda_regression_diagnostic.png)\n')
report.append('## Adjusted associations on BRINDA-corrected outcomes\n')
report.append(brinda_table.to_markdown(index=False) + '\n')
report.append('## BRINDA-corrected iron deficiency by feeding stratum\n')
report.append(strat_table.to_markdown(index=False) + '\n')
report.append('![Stratified by feeding](figures/brinda_stratified_by_feeding.png)\n')
report.append('## Iron × breastfeeding interaction on BRINDA-corrected iron deficiency\n')
report.append(interaction_brinda_table.to_markdown(index=False) + '\n')
report.append('## Negative-control sensitivity — accidental hospitalisation\n')
if not negative_control_table.empty:
    report.append(negative_control_table.to_markdown(index=False) + '\n')
else:
    report.append('Fit failed (insufficient events).\n')
report.append('---\n')
report.append('## Notes\n')
report.append('- BRINDA correction uses CRP only because AGP is not measured in ENANI-2019.\n')
report.append('- Stratified PRs are analytical (HC1) without bootstrap; the formal interaction test is bootstrapped.\n')
report.append('- A null negative-control result supports the design; a non-null result would flag residual confounding.\n')
(OUTPUT_DIR / 'REPORT.md').write_text('\n'.join(report))

print(f'Outputs written to: {OUTPUT_DIR.resolve()}')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(f'  {path.relative_to(OUTPUT_DIR)}  ({path.stat().st_size/1024:.1f} KB)')
